# RAG-Based Customer Support Assistant

In [1]:
import ollama

print("Ollama imported successfully")

Ollama imported successfully


In [9]:
import sys
print(sys.executable)

c:\Users\pairkshith\VS Code projects\RAG_based_Customer_Support_Assistance\venv\Scripts\python.exe


In [2]:
import sys
import os

# Add project root to path
sys.path.append(os.getcwd())

In [1]:
import os
import shutil

from modules.loader import load_pdf
from modules.chunker import chunk_documents
from modules.embedder import get_embedding_model
from modules.vector_store import create_vector_store
from modules.retriever import get_retriever
from modules.generator import get_llm, generate_answer
from modules.graph import build_graph
from modules.router import route

c:\Users\pairkshith\VS Code projects\RAG_based_Customer_Support_Assistance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def list_pdfs():
    files = os.listdir("data")
    pdfs = [f for f in files if f.endswith(".pdf")]
    
    print("📂 Available PDFs:\n")
    for i, pdf in enumerate(pdfs):
        print(f"{i+1}. {pdf}")
    
    return pdfs

pdfs = list_pdfs()

📂 Available PDFs:

1. knowledge.pdf
2. Waste Classifier.pdf


In [4]:
choice = int(input("\nEnter PDF number: ")) - 1
selected_file = f"data/{pdfs[choice]}"

print(f"\n✅ Selected File: {selected_file}")


✅ Selected File: data/knowledge.pdf


In [5]:
docs = load_pdf(selected_file)

print("✅ PDF Loaded")
print("Pages:", len(docs))

✅ PDF Loaded
Pages: 8


In [6]:
chunks = chunk_documents(docs)

print("✅ Chunking Done")
print("Total Chunks:", len(chunks))

✅ Chunking Done
Total Chunks: 15


In [7]:
# Clear old DB
if os.path.exists("chroma_db"):
    shutil.rmtree("chroma_db")

embedding_model = get_embedding_model()
db = create_vector_store(chunks, embedding_model)

print("✅ Vector DB Created")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2108.19it/s]


✅ Vector DB Created


c:\Users\pairkshith\VS Code projects\RAG_based_Customer_Support_Assistance\modules\vector_store.py:12: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [8]:
retriever = get_retriever(db)

print("✅ Retriever Ready")

✅ Retriever Ready


In [9]:
llm = get_llm()

print("✅ LLM Ready (Llama3)")

✅ LLM Ready (Llama3)


In [10]:
app = build_graph(llm, retriever)

print("✅ LangGraph Workflow Ready")

✅ LangGraph Workflow Ready


In [11]:
query = "What documents have to be submitted with the application?"

result = app.invoke({"query": query})

print("🧑 User Query:", query)
print("\n🤖 Bot Response:\n")
print(result["output"])

🧑 User Query: What documents have to be submitted with the application?

🤖 Bot Response:

The COMPLETE answer from the context is:

The following documents shall be enclosed for each applicant: 

1. Proof of present citizenship 

2. Evidence of self or parents or grand parents, 

(a)  being eligible to become a citizen of India at the time of 
commencement of the Constitution; or 
(b)  belonging to a territory that became a part of India after 
15 th August, 1947; or 
(c)  being a citizen of India on or after 26 th January, 1950 

These could be: 
 
(i)  Copy of the passport :or 
(ii) Copy of the domicile certificate issued by the Competent 
authority ;or
in local currency for each PIO card holder).


In [12]:
docs = retriever.invoke(query)

for i, doc in enumerate(docs[:3]):
    print(f"\n--- Retrieved Chunk {i+1} ---\n")
    print(doc.page_content[:300])


--- Retrieved Chunk 1 ---

6.  In what form should a person  apply for an OCI and where are the 
forms available? 
 
A family consisting of spouses and upto two minor children can apply 
in the same form i.e. Form XIX, which can be filed online or downloaded 
from our website www.mha.nic.in. 
 
7.  Can application form be fil

--- Retrieved Chunk 2 ---

in local currency for each PIO card holder).  In case of 
application filled in India, fee Rs.14,230/- for general category 
and for PIO card holders Rs.1,290/- to be paid by way of 
Demand Draft. 
 
5.  PIO card holders should submit a copy of his/her PIO card. 
 
9.  What documents would qualify f

--- Retrieved Chunk 3 ---

12.  Whether the applicant (s) have to take oath before the Counsel of 
the Indian Mission/Post? 
 
No. Earlier provision in this regard has been done away with. 
 
13.  Where to submit the application? 
 
To the Indian Mission/ Post of the country of citizenship of the 
applicant. If the applicant 


In [13]:
decision = route(query, docs, result["output"])

print("Routing Decision:", decision)

Routing Decision: answer


In [14]:
while True:
    try:
        query = input("\nYou: ")
        
        if query.lower() in ["exit", "quit"]:
            print("👋 Exiting chatbot")
            break
        
        result = app.invoke({"query": query})
        print("\nBot:", result["output"])

    except KeyboardInterrupt:
        print("\n👋 Chat stopped manually")
        break


Bot: 12.  Whether the applicant (s) have to take oath before the Counsel of 
the Indian Mission/Post? 
 
No. Earlier provision in this regard has been done away with.
👋 Exiting chatbot
